# Load & Preprocess the Data

In [4]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('edited2_measures.csv', delimiter=';')
# Convert string numbers with commas to floats
for column in df.columns:
    if df[column].dtype == 'object':
        try:
            df[column] = df[column].str.replace(',', '.').astype(float)
        except ValueError:
            pass  # If conversion fails, it's probably a categorical column
X = df.drop(columns=['activity'])
y = df['activity']

# Split the data 0.7 train & 0.3 test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Configure & Train the Models
## Random Forest

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Set random seed for reproducibility
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest Accuracy: {accuracy_rf:.4f}')

Random Forest Accuracy: 0.9837


## XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Convert categorical variables in X to numerical variables using one-hot encoding
L = pd.get_dummies(X)

# Encode the target labels
label_encoder = LabelEncoder()
Y_encoded = label_encoder.fit_transform(y)

# Split the data into training and testing sets
L_train, L_test, n_train, n_test = train_test_split(L, Y_encoded, test_size=0.2, random_state=42)



# Set random seef for reproducibility
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb.fit(L_train, n_train)
y_pred_xgb = xgb.predict(L_test)
accuracy_xgb = accuracy_score(n_test, y_pred_xgb)
print(f'XGBoost Accuracy: {accuracy_xgb:.4f}')

## Stacking Classifier

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Set random seed for reproducibility
estimators = [('rf', RandomForestClassifier(random_state=42)),
              ('svc', SVC(probability=True, random_state=42))]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5
)
stacking_clf.fit(X_train, y_train)
y_pred_stacking = stacking_clf.predict(X_test)
accuracy_stacking = accuracy_score(y_test, y_pred_stacking)
print(f"Stacking Classifier Accuracy: {accuracy_stacking:.4f}")

# Identify the Best Accuracy

In [6]:
accuracies = {
    'Random Forest': accuracy_rf,
    'XGBoost': accuracy_xgb,
    'Stacking Classifier': accuracy_stacking
}
best_model = max(accuracies, key=accuracies.get)
best_accuracy = accuracies[best_model]
print(f"Best Model: {best_model} with Accuracy: {best_accuracy:.4f}")

Best Model: XGBoost with Accuracy: 0.9946
